# Relatório - Visão Computacional e Aprendizado Profundo


## Programa de Pós-Graduação em Informática (PPGIa) / PUCPR (2026-2)

**Prof. Rayson Laroca**

`Thiago Costa` - `thiagohesc@gmail.com`
`https://github.com/thiagohesc/cub-200-2011-bird`

# Importe as bibliotecas necessárias

In [ ]:
import ctypes
import math
import re
from pathlib import Path
from time import perf_counter

import albumentations as A
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps, ImageStat
from skimage.color import rgb2hsv
from skimage.feature import local_binary_pattern
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tqdm.auto import tqdm

## Configurações GPU

In [ ]:
caminho_cusolver = (
    Path(tf.__file__).resolve().parent.parent
    / "nvidia"
    / "cusolver"
    / "lib"
    / "libcusolver.so.11"
)
if caminho_cusolver.is_file():
    try:
        biblioteca_cusolver = ctypes.CDLL(str(caminho_cusolver))
    except OSError as erro:
        print("Falha ao carregar a biblioteca cuSOLVER da .venv:", erro)

gpus = tf.config.list_physical_devices("GPU")

try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError as erro:
    print("Reinicie o kernel para configurar o crescimento de memória da GPU:", erro)

mixed_precision.set_global_policy("mixed_float16" if gpus else "float32")

print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("Política:", mixed_precision.global_policy())

# 1. Defina a tarefa e o conjunto de dados

## Problema de Classificação

O projeto tem como objetivo desenvolver e avaliar métodos de **classificação de espécies de aves a partir de imagens**, utilizando: abordagens tradicionais de processamento de imagens, aprendizado de máquina e aprendizado profundo.

## Conjunto de Dados

Será utilizado o conjunto de dados **Caltech-UCSD Birds-200-2011 (CUB-200-2011)**, desenvolvido para pesquisas relacionadas à classificação visual de espécies de aves.

## Características do Conjunto de Dados

- **Número de classes:** 200 espécies de aves;
- **Número total de imagens:** 11.788;
- **Localizações de partes por imagem:** 15;
- **Atributos binários por imagem:** 312;
- **Caixa delimitadora (*Bounding Box*) por imagem:** 1.

## Objetivo da Classificação

O objetivo principal é desenvolver modelos capazes de receber como entrada uma imagem contendo uma ave e identificar corretamente **a qual das 200 espécies disponíveis no conjunto de dados ela pertence**.

## Fonte do Conjunto de Dados

O conjunto de dados utilizado neste projeto está disponível publicamente no Kaggle:

https://www.kaggle.com/datasets/wenewone/cub2002011

Informações detalhadas sobre o conjunto de dados original podem ser encontradas no relatório técnico disponibilizado pelo Caltech:

https://authors.library.caltech.edu/27452/

# 2. Aquisição do conjunto de dados

### Obtenção e preparação dos dados

O conjunto foi obtido pelo Kaggle e validado a partir dos arquivos oficiais de classes, imagens, rótulos, divisão de treino/teste e bounding boxes. Essas informações foram reunidas em uma única tabela de metadados, utilizada para verificar identificadores, relacionamentos, dimensões e disponibilidade das imagens.

Nesta etapa, foram mantidas todas as **11.788 imagens** em suas resoluções originais, sem subamostragem ou redimensionamento. As imagens foram lidas individualmente apenas para extração de informações, evitando carregar todo o conjunto na memória.

As transformações necessárias para o treinamento dos modelos foram definidas posteriormente, na Etapa 5.

O CUB-200-2011 é adequado para este projeto por apresentar um problema de classificação **fine-grained**, no qual espécies diferentes podem ser visualmente semelhantes, enquanto imagens da mesma espécie apresentam variações de pose, iluminação e fundo. Isso permite comparar abordagens tradicionais com modelos de deep learning.

In [ ]:
"""
Obtenção do conjunto de dados pelo Kaggle
Se True, o código fará o download do conjunto de dados CUB-200-2011 do Kaggle e o salvará no diretório ./data/raw.
"""
DOWNLOAD = True

# Definir uma semente para reprodutibilidade Global
SEED = 42

if DOWNLOAD:
    DATA_DIR = Path("./data/raw")
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    dataset_path = kagglehub.dataset_download(
        "wenewone/cub2002011", output_dir=str(DATA_DIR)
    )

    print(f"Dataset disponível em: {dataset_path}")

"""
Extrair metadados do conjunto de dados CUB-200-2011 e salvar em um arquivo CSV.
O arquivo CSV conterá informações sobre as imagens, classes, rótulos, divisão entre treino e teste, e bounding boxes.
"""
# Classes
df_classes = pd.read_csv(
    DATA_DIR / "CUB_200_2011" / "classes.txt",
    sep=r"\s+",
    names=["class_id", "class_name"],
)

# Imagens
df_images = pd.read_csv(
    DATA_DIR / "CUB_200_2011" / "images.txt",
    sep=r"\s+",
    names=["image_id", "image_path"],
)

# Relação entre imagens e classes
df_labels = pd.read_csv(
    DATA_DIR / "CUB_200_2011" / "image_class_labels.txt",
    sep=r"\s+",
    names=["image_id", "class_id"],
)

# Divisão entre treino e teste (1 = treino; 0 = teste)
df_split = pd.read_csv(
    DATA_DIR / "CUB_200_2011" / "train_test_split.txt",
    sep=r"\s+",
    names=["image_id", "image_split"],
)

# Bounding boxes
df_bounding_boxes = pd.read_csv(
    DATA_DIR / "CUB_200_2011" / "bounding_boxes.txt",
    sep=r"\s+",
    names=[
        "image_id",
        "bbox_x",
        "bbox_y",
        "bbox_width",
        "bbox_height",
    ],
)

df_metadados = (
    df_images.merge(
        df_labels,
        on="image_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_split,
        on="image_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_bounding_boxes,
        on="image_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_classes,
        on="class_id",
        how="left",
        validate="many_to_one",
    )
)

df_metadados = df_metadados.astype(
    {
        "image_id": "int32",
        "image_path": "string",
        "class_id": "int16",
        "image_split": "int8",
        "bbox_x": "float32",
        "bbox_y": "float32",
        "bbox_width": "float32",
        "bbox_height": "float32",
        "class_name": "string",
    },
)

df_metadados.to_csv(
    DATA_DIR / "CUB_200_2011" / "metadados.csv",
    index=False,
)

print(f"Metadados disponíveis em: {DATA_DIR / 'CUB_200_2011' / 'metadados.csv'}")
df_metadados.head()

# 3. Análise e visualização do conjunto de dados

### Análise do conjunto de dados

O conjunto apresenta uma distribuição relativamente equilibrada entre as 200 classes. A quantidade de imagens por espécie varia de **41 a 60**, com média de **58,94** e mediana de **60 imagens**, reduzindo o risco de favorecer classes com maior quantidade de exemplos.

As imagens apresentam variações de resolução, com larguras entre **121 e 500 pixels** e alturas entre **120 e 500 pixels**. Imagens menores podem dificultar a identificação de detalhes importantes, como plumagem, formato do bico e padrões das asas.

Também foram observadas diferenças de iluminação e na área ocupada pela ave. Imagens muito claras ou escuras podem esconder características relevantes, enquanto aves pequenas na cena aumentam a influência do fundo durante o aprendizado.

Como o CUB-200-2011 é um problema de classificação **fine-grained**, a semelhança visual entre as espécies é um dos principais desafios. Pequenas diferenças podem determinar a classe correta, enquanto fatores como pose, escala, iluminação e oclusões podem dificultar essa distinção.

Na análise realizada e na inspeção visual das amostras, não foram identificados indícios evidentes de desequilíbrio significativo, problemas sistemáticos de qualidade ou erros de rotulagem.

### Estatísticas básicas

In [ ]:
# Validação e numero total de imagens
imagens_existentes = df_metadados["image_path"].apply(
    lambda path: (DATA_DIR / "CUB_200_2011" / "images" / path).is_file()
)
total_imagens = len(df_metadados)
print(f"Total de imagens existentes: {imagens_existentes.sum()} de {total_imagens}")


# Extrair informações das imagens
def extrair_informacoes_imagem(image_path):
    path = DATA_DIR / "CUB_200_2011" / "images" / image_path
    with Image.open(path) as image:
        return pd.Series(
            {
                "image_width": image.width,
                "image_height": image.height,
                "image_format": image.format,
            }
        )


df_image_info = df_metadados["image_path"].apply(extrair_informacoes_imagem)

df_metadados = pd.concat(
    [df_metadados, df_image_info],
    axis=1,
)

# Faixas de largura e altura das imagens
largura_min = df_metadados["image_width"].min()
largura_max = df_metadados["image_width"].max()

altura_min = df_metadados["image_height"].min()
altura_max = df_metadados["image_height"].max()

print("Largura mínima:", largura_min)
print("Largura máxima:", largura_max)

print("Altura mínima:", altura_min)
print("Altura máxima:", altura_max)

# Validação e numero total de classes
classes_existentes = df_metadados.loc[imagens_existentes, "class_id"].unique()
total_classes = df_metadados["class_id"].nunique()
print(f"Total de classes existentes: {len(classes_existentes)} de {total_classes}")

In [ ]:
# Plotar estatísticas básicas do conjunto de dados
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].axis("off")

texto = (
    f"Total de imagens\n"
    f"{total_imagens}\n\n"
    f"Número de classes\n"
    f"{total_classes}\n\n"
    f"Faixa de largura\n"
    f"{largura_min} - {largura_max} px\n\n"
    f"Faixa de altura\n"
    f"{altura_min} - {altura_max} px"
)

axes[0, 0].text(
    0.5,
    0.5,
    texto,
    horizontalalignment="center",
    verticalalignment="center",
    fontsize=14,
)

axes[0, 0].set_title("Estatísticas gerais")

axes[0, 1].hist(df_metadados["image_width"], bins=30, edgecolor="black")

axes[0, 1].set_title("Distribuição das larguras")
axes[0, 1].set_xlabel("Largura (pixels)")
axes[0, 1].set_ylabel("Quantidade de imagens")

axes[1, 0].hist(df_metadados["image_height"], bins=30, edgecolor="black")

axes[1, 0].set_title("Distribuição das alturas")
axes[1, 0].set_xlabel("Altura (pixels)")
axes[1, 0].set_ylabel("Quantidade de imagens")

axes[1, 1].scatter(df_metadados["image_width"], df_metadados["image_height"], alpha=0.4)

axes[1, 1].set_title("Distribuição das resoluções")
axes[1, 1].set_xlabel("Largura (pixels)")
axes[1, 1].set_ylabel("Altura (pixels)")


plt.suptitle("Estatísticas básicas do conjunto de dados", fontsize=16)

plt.tight_layout()
plt.show()

### Distribuição das classes

In [ ]:
# Distribuição de imagens por classe
distribuicao_classes = df_metadados["class_id"].value_counts().sort_index()
distribuicao_classes.describe()

In [ ]:
# Plotar distribuição de imagens por classe
distribuicao_classes = df_metadados["class_id"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))


# Quantidade de imagens por classe
axes[0].bar(distribuicao_classes.index, distribuicao_classes.values)

axes[0].set_title("Quantidade de imagens por classe")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Quantidade de imagens")


# Histograma
axes[1].hist(distribuicao_classes.values, bins=15, edgecolor="black")

axes[1].set_title("Distribuição do número de imagens")
axes[1].set_xlabel("Quantidade de imagens por classe")
axes[1].set_ylabel("Número de classes")


plt.tight_layout()
plt.show()

### Exemplos

In [ ]:
# Função para exibir amostras de imagens com bounding boxes
def exibir_amostras(amostras, titulo, colunas=4):
    if amostras.empty:
        print("Nenhuma imagem para exibir.")
        return

    linhas = math.ceil(len(amostras) / colunas)

    fig, axes = plt.subplots(
        linhas, colunas, figsize=(4 * colunas, 3.5 * linhas), squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    # Exibe as imagens
    for ax, (_, registro) in zip(axes.flat, amostras.iterrows()):
        try:
            with Image.open(
                DATA_DIR / "CUB_200_2011" / "images" / registro["image_path"]
            ) as imagem:
                ax.imshow(imagem.convert("RGB"))

            ax.add_patch(
                Rectangle(
                    (registro["bbox_x"], registro["bbox_y"]),
                    registro["bbox_width"],
                    registro["bbox_height"],
                    fill=False,
                    edgecolor="lime",
                    linewidth=1.5,
                )
            )

            nome = re.sub(r"^\d+\.", "", registro["class_name"]).replace("_", " ")

            ax.set_title(
                f"ID {registro['image_id']} | Class ID {registro['class_id']}\n{nome}",
                fontsize=9,
            )

        except (OSError, ValueError) as erro:
            ax.set_title(f"ID {registro['image_id']}: {type(erro).__name__}")

    fig.suptitle(titulo, fontsize=14)

    plt.tight_layout()
    plt.show()


# Exemplos de classes diferentes
classes_amostra = (
    df_metadados["class_id"].drop_duplicates().sample(12, random_state=SEED)
)
amostras = pd.concat(
    [
        df_metadados[df_metadados["class_id"] == classe].sample(1, random_state=SEED)
        for classe in classes_amostra
    ]
)

exibir_amostras(amostras, "Amostras de 12 classes diferentes")

In [ ]:
# Exemplos de variação intra-classe
classe_amostra = (
    df_metadados["class_id"].drop_duplicates().sample(1, random_state=SEED).iloc[0]
)

amostras = df_metadados[df_metadados["class_id"] == classe_amostra].sample(
    12, random_state=SEED
)

exibir_amostras(amostras, "Variação intra-classe")

In [ ]:
# Exemplos de classes visualmente similares
classes_comparacao = [1, 2, 3, 4]

amostras = pd.concat(
    [
        df_metadados[df_metadados["class_id"] == classe].sample(
            min(4, len(df_metadados[df_metadados["class_id"] == classe])),
            random_state=SEED,
        )
        for classe in classes_comparacao
    ]
)

exibir_amostras(amostras, "Similaridade inter-classe", colunas=4)

In [ ]:
def calcular_iluminacao(image_path):
    try:
        caminho = DATA_DIR / "CUB_200_2011" / "images" / image_path

        with Image.open(caminho) as imagem:
            imagem_cinza = imagem.convert("L")
            return ImageStat.Stat(imagem_cinza).mean[0]

    except (OSError, ValueError):
        return None


# Iluminação
df_metadados["brightness"] = df_metadados["image_path"].apply(calcular_iluminacao)


# Resolução da imagem
df_metadados["resolution"] = df_metadados["image_width"] * df_metadados["image_height"]


# Área da bounding box
df_metadados["bbox_area"] = df_metadados["bbox_width"] * df_metadados["bbox_height"]


# Área da imagem
df_metadados["image_area"] = df_metadados["image_width"] * df_metadados["image_height"]


# Proporção da imagem ocupada pela bounding box
df_metadados["bird_ratio"] = df_metadados["bbox_area"] / df_metadados["image_area"]


df_brightness = df_metadados.dropna(subset=["brightness"])

menor_iluminacao = df_brightness.loc[df_brightness["brightness"].idxmin()].copy()

maior_iluminacao = df_brightness.loc[df_brightness["brightness"].idxmax()].copy()

menor_resolucao = df_metadados.loc[df_metadados["resolution"].idxmin()].copy()

maior_resolucao = df_metadados.loc[df_metadados["resolution"].idxmax()].copy()

menor_bbox = df_metadados.loc[df_metadados["bird_ratio"].idxmin()].copy()

maior_bbox = df_metadados.loc[df_metadados["bird_ratio"].idxmax()].copy()


# Identificar cada exemplo
menor_iluminacao["analise"] = "Menor iluminação"
maior_iluminacao["analise"] = "Maior iluminação"

menor_resolucao["analise"] = "Menor resolução"
maior_resolucao["analise"] = "Maior resolução"

menor_bbox["analise"] = "Menor bbox"
maior_bbox["analise"] = "Maior bbox"


# Criar DataFrame com os casos extremos
amostras_extremos = pd.DataFrame(
    [
        menor_iluminacao,
        maior_iluminacao,
        menor_resolucao,
        maior_resolucao,
        menor_bbox,
        maior_bbox,
    ]
)

amostras_extremos[
    [
        "analise",
        "image_id",
        "class_name",
        "image_width",
        "image_height",
        "brightness",
        "resolution",
        "bird_ratio",
    ]
]

exibir_amostras(
    amostras_extremos[
        amostras_extremos["analise"].isin(["Menor iluminação", "Maior iluminação"])
    ],
    (
        f"Casos extremos de iluminação | "
        f"min: {df_metadados['brightness'].min():.2f} | "
        f"max: {df_metadados['brightness'].max():.2f}"
    ),
    colunas=2,
)

In [ ]:
exibir_amostras(
    amostras_extremos[
        amostras_extremos["analise"].isin(["Menor resolução", "Maior resolução"])
    ],
    (
        f"Casos extremos de resolução | "
        f"min: {int(df_metadados['resolution'].min())} px | "
        f"max: {int(df_metadados['resolution'].max())} px"
    ),
    colunas=2,
)

In [ ]:
exibir_amostras(
    amostras_extremos[amostras_extremos["analise"].isin(["Menor bbox", "Maior bbox"])],
    (
        f"Casos extremos de bounding box | "
        f"min: {df_metadados['bird_ratio'].min():.1%} | "
        f"max: {df_metadados['bird_ratio'].max():.1%}"
    ),
    colunas=2,
)

### Divisão do conjunto de dados

O CUB-200-2011 possui uma divisão oficial com **5.994 imagens para treino** e **5.794 para teste**, distribuídas entre 200 classes.

Como o dataset não possui um conjunto de validação separado, **20% do conjunto oficial de treino** foi reservado para validação por meio de uma divisão estratificada por classe. Os 80% restantes foram utilizados para treinamento.

O conjunto oficial de teste foi mantido separado e utilizado somente na avaliação final dos modelos.

In [ ]:
df_metadados["split"] = df_metadados["image_split"].map({1: "Treino", 0: "Teste"})
df_metadados["aspect_ratio"] = (
    df_metadados["image_width"] / df_metadados["image_height"]
)

resumo = pd.Series(
    {
        "Imagens": len(df_metadados),
        "Classes": df_metadados["class_id"].nunique(),
        "Treino": (df_metadados["split"] == "Treino").sum(),
        "Teste": (df_metadados["split"] == "Teste").sum(),
        "Formatos distintos": df_metadados["image_format"].nunique(),
        "Valores ausentes": df_metadados.isna().sum().sum(),
    },
    name="Quantidade",
)

# Quantidade de imagens por classe e por conjunto
por_split = pd.crosstab(df_metadados["class_name"], df_metadados["split"])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

altura = 0.35

# 10 classes com menos imagens de treino
menos_treino = por_split.nsmallest(10, "Treino")

y = np.arange(len(menos_treino))

axes[0].barh(y - altura / 2, menos_treino["Treino"], height=altura, label="Treino")

axes[0].barh(y + altura / 2, menos_treino["Teste"], height=altura, label="Teste")

axes[0].set_yticks(y)
axes[0].set_yticklabels(menos_treino.index)

axes[0].set(
    title="10 classes com menos imagens de treino",
    xlabel="Quantidade de imagens",
    ylabel="Classe",
)

axes[0].invert_yaxis()
axes[0].legend()

# 10 classes com menos imagens de teste
menos_teste = por_split.nsmallest(10, "Teste")

y = np.arange(len(menos_teste))

axes[1].barh(y - altura / 2, menos_teste["Treino"], height=altura, label="Treino")

axes[1].barh(y + altura / 2, menos_teste["Teste"], height=altura, label="Teste")

axes[1].set_yticks(y)
axes[1].set_yticklabels(menos_teste.index)

axes[1].set(
    title="10 classes com menos imagens de teste",
    xlabel="Quantidade de imagens",
    ylabel="Classe",
)

axes[1].invert_yaxis()
axes[1].legend()


plt.tight_layout()
plt.show()

# 4. Divisão do conjunto de dados

Como o dataset não possui um conjunto de validação separado, **20% do conjunto oficial de treino** foi reservado para essa finalidade. A divisão foi realizada de forma estratificada por classe, mantendo a proporção das espécies entre treino e validação.

O conjunto oficial de teste foi preservado e utilizado somente na avaliação final dos modelos.1

In [ ]:
indices_treino = df_metadados[df_metadados["image_split"] == 1].index

_, indices_validacao = train_test_split(
    indices_treino,
    test_size=0.20,
    random_state=SEED,
    stratify=df_metadados.loc[indices_treino, "class_id"],
)

df_metadados.loc[indices_validacao, "image_split"] = 2

df_metadados["split"] = df_metadados["image_split"].map(
    {0: "Teste", 1: "Treino", 2: "Validação"}
)

df_metadados.to_csv(
    DATA_DIR / "CUB_200_2011" / "metadados.csv",
    index=False,
)
df_metadados["split"].value_counts()

# 5. Pré-processamento de imagens

As imagens foram convertidas para RGB e padronizadas para **384 × 384 pixels**. Como o conjunto possui diferentes resoluções e proporções, o redimensionamento foi realizado preservando a proporção original e utilizando preenchimento quando necessário. Dessa forma, evitam-se distorções nas características das aves.

A resolução de **384 × 384** foi escolhida para preservar detalhes importantes, como plumagem, bico e padrões das asas, mantendo um custo computacional adequado ao hardware e aos modelos de deep learning utilizados.

No conjunto de treino também foi criada uma segunda versão de cada imagem utilizando a **bounding box** disponibilizada pelo dataset. Assim, o treinamento pode utilizar tanto a imagem completa, que preserva o contexto da cena, quanto o recorte da ave, que reduz a influência do fundo e destaca suas características visuais.

Os conjuntos de validação e teste foram mantidos apenas com imagens completas, sem essas variações, permitindo avaliar os modelos em condições mais próximas das imagens originais.

In [ ]:
# Pre-processamento das imagens: redimensionamento e salvamento em diretórios separados para treino, validação e teste.
IMG_SIZE = (384, 384)

PROCESSED_DIR = Path("./data/processed")
IMAGES_DIR = DATA_DIR / "CUB_200_2011" / "images"
TRAIN_FULL_DIR = PROCESSED_DIR / "train" / "full"
TRAIN_BBOX_DIR = PROCESSED_DIR / "train" / "bbox"

VALIDATION_FULL_DIR = PROCESSED_DIR / "validation" / "full"

TEST_FULL_DIR = PROCESSED_DIR / "test" / "full"


def redimensionar_imagem(imagem, tamanho=IMG_SIZE):
    """
    Redimensiona uma imagem para o tamanho especificado.
    """
    return ImageOps.pad(
        imagem, tamanho, method=Image.Resampling.LANCZOS, color=(0, 0, 0)
    )


def processar_full(caminho_entrada, caminho_saida):
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(caminho_entrada) as imagem:
        imagem = imagem.convert("RGB")

        imagem = redimensionar_imagem(imagem)

        imagem.save(caminho_saida, quality=95)


def recortar_bbox(imagem, registro):
    largura_imagem, altura_imagem = imagem.size

    x1 = int(registro["bbox_x"])
    y1 = int(registro["bbox_y"])

    x2 = int(registro["bbox_x"] + registro["bbox_width"])

    y2 = int(registro["bbox_y"] + registro["bbox_height"])

    x1 = max(0, x1)
    y1 = max(0, y1)

    x2 = min(largura_imagem, x2)
    y2 = min(altura_imagem, y2)

    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Bounding box inválida para a imagem {registro['image_id']}")

    return imagem.crop((x1, y1, x2, y2))


def processar_bbox(registro, caminho_entrada, caminho_saida):
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(caminho_entrada) as imagem:
        imagem = imagem.convert("RGB")

        imagem = recortar_bbox(imagem, registro)

        imagem = redimensionar_imagem(imagem)

        imagem.save(caminho_saida, quality=95)


registros_processados = []

for _, registro in df_metadados.iterrows():
    caminho_original = IMAGES_DIR / registro["image_path"]

    # TREINO
    if registro["split"] == "Treino":
        caminho_full = TRAIN_FULL_DIR / registro["image_path"]

        caminho_bbox = TRAIN_BBOX_DIR / registro["image_path"]

        processar_full(caminho_original, caminho_full)

        processar_bbox(registro, caminho_original, caminho_bbox)

        registros_processados.append(
            {
                "image_id": registro["image_id"],
                "class_id": registro["class_id"],
                "class_name": registro["class_name"],
                "split": "Treino",
                "variant": "full",
                "image_path": str(caminho_full.relative_to(PROCESSED_DIR)),
            }
        )

        registros_processados.append(
            {
                "image_id": registro["image_id"],
                "class_id": registro["class_id"],
                "class_name": registro["class_name"],
                "split": "Treino",
                "variant": "bbox",
                "image_path": str(caminho_bbox.relative_to(PROCESSED_DIR)),
            }
        )

    # VALIDAÇÃO
    elif registro["split"] == "Validação":
        caminho_full = VALIDATION_FULL_DIR / registro["image_path"]

        processar_full(caminho_original, caminho_full)

        registros_processados.append(
            {
                "image_id": registro["image_id"],
                "class_id": registro["class_id"],
                "class_name": registro["class_name"],
                "split": "Validação",
                "variant": "full",
                "image_path": str(caminho_full.relative_to(PROCESSED_DIR)),
            }
        )

    # TESTE
    elif registro["split"] == "Teste":
        caminho_full = TEST_FULL_DIR / registro["image_path"]

        processar_full(caminho_original, caminho_full)

        registros_processados.append(
            {
                "image_id": registro["image_id"],
                "class_id": registro["class_id"],
                "class_name": registro["class_name"],
                "split": "Teste",
                "variant": "full",
                "image_path": str(caminho_full.relative_to(PROCESSED_DIR)),
            }
        )

df_processado = pd.DataFrame(registros_processados)
df_processado.to_csv(PROCESSED_DIR / "metadados.csv", index=False)
df_processado.head()

In [ ]:
# Conferindo a distribuição de imagens por conjunto e variante
pd.crosstab(df_processado["split"], df_processado["variant"])

Não foram aplicadas conversões para escala de cinza, filtros de redução de ruído ou correções fixas de brilho e contraste. Essas decisões foram tomadas porque a cor, a textura e pequenos detalhes visuais são importantes para diferenciar espécies semelhantes, e transformações desse tipo poderiam eliminar informações relevantes.

A normalização dos pixels também não foi aplicada diretamente às imagens salvas. Esse processamento será realizado durante o treinamento, de acordo com os requisitos de cada modelo.

Transformações como rotação, variações de brilho e contraste e espelhamento foram reservadas para a etapa de aumento de dados e aplicadas somente ao conjunto de treino.

# 6. Abordagem tradicional


### Preparação dos conjuntos

Os metadados processados foram carregados e, para esta etapa, foram selecionadas apenas as imagens da variante `full`. Em seguida, os dados foram separados em treino, validação e teste de acordo com a divisão definida anteriormente.

In [ ]:
SEED = 42
PROCESSED_DIR = Path("./data/processed")

caminho_metadados = PROCESSED_DIR / "metadados.csv"
if not caminho_metadados.is_file():
    raise FileNotFoundError(
        "Metadados processados não encontrados. Execute a etapa 5 primeiro."
    )

df_processado = pd.read_csv(caminho_metadados)
df_full = df_processado[df_processado["variant"] == "full"].copy()

df_train = df_full[df_full["split"] == "Treino"].copy()
df_val = df_full[df_full["split"] == "Validação"].copy()
df_test = df_full[df_full["split"] == "Teste"].copy()

# Resumo dos conjuntos de dados
resumo_conjuntos = pd.DataFrame(
    {
        "Conjunto": ["Treino", "Validação", "Teste"],
        "Imagens": [len(df_train), len(df_val), len(df_test)],
        "Classes": [
            df_train["class_id"].nunique(),
            df_val["class_id"].nunique(),
            df_test["class_id"].nunique(),
        ],
    }
)
display(resumo_conjuntos)

### Extração de características

Para cada imagem em **384 × 384 pixels**, foram extraídos dois vetores de características: **10 atributos de textura com LBP** e **64 atributos de cor com HSV**.

Como os dois descritores resumem a imagem inteira em histogramas, informações do fundo também influenciam os atributos e a posição das características na imagem não é preservada.

A extração é realizada diretamente a partir das imagens, e as matrizes geradas permanecem em memória para serem utilizadas no treinamento dos classificadores.

### Parâmetros e leitura das imagens

Foram definidos os parâmetros utilizados pelos descritores e criada uma função para carregar as imagens em RGB, padronizar suas dimensões e convertê-las para arrays NumPy. Essa etapa garante que todas as imagens sejam processadas de forma consistente antes da extração das características.

In [ ]:
FEATURE_SIZE = (384, 384)
LBP_POINTS = 8
LBP_RADIUS = 1
HSV_BINS = (8, 8)


def carregar_imagem(caminho):
    with Image.open(caminho) as imagem:
        imagem = imagem.convert("RGB").resize(
            FEATURE_SIZE, resample=Image.Resampling.LANCZOS
        )
        return np.asarray(imagem)


### Textura com LBP

Para representar a textura das imagens, foi utilizado o descritor **LBP uniforme**, com 8 vizinhos e raio de 1 pixel, aplicado sobre a imagem em escala de cinza.

O histograma resultante possui **10 atributos por imagem** e foi normalizado pela soma das contagens, fazendo com que seus valores totalizem 1. Essa normalização permite comparar os padrões de textura entre imagens de forma mais consistente.

In [ ]:
def extrair_lbp(imagem):
    cinza = np.asarray(Image.fromarray(imagem).convert("L"))
    lbp = local_binary_pattern(cinza, P=LBP_POINTS, R=LBP_RADIUS, method="uniform")

    histograma, _ = np.histogram(lbp, bins=np.arange(LBP_POINTS + 3))
    histograma = histograma.astype(np.float32)
    return histograma / max(histograma.sum(), 1e-7)

### Cor com HSV

Para representar as informações de cor das imagens, foi utilizado um histograma conjunto dos canais de matiz (H) e saturação (S) do espaço de cores HSV. Foram utilizadas 8 divisões em cada canal, resultando em **64 atributos por imagem**.

O canal de brilho (V) não foi incluído no histograma para reduzir a influência direta das variações de iluminação. Ao final, o vetor de características foi normalizado pela norma L2, permitindo uma comparação mais consistente entre as imagens.

In [ ]:
def extrair_hsv(imagem):
    hsv = rgb2hsv(imagem)
    histograma, _, _ = np.histogram2d(
        hsv[:, :, 0].ravel(),
        hsv[:, :, 1].ravel(),
        bins=HSV_BINS,
        range=((0, 1), (0, 1)),
    )

    histograma = histograma.astype(np.float32).ravel()
    return histograma / max(np.linalg.norm(histograma), 1e-7)

### Extração para um conjunto de imagens

As imagens foram percorridas a partir dos caminhos registrados nos metadados e, para cada uma, foram extraídos os descritores **LBP** e **HSV**. As matrizes `X_lbp` e `X_hsv`, assim como o vetor `y`, mantêm a mesma ordem das imagens processadas.

Os rótulos das classes também foram convertidos do intervalo **1–200** para **0–199**, mantendo compatibilidade com os classificadores utilizados nas etapas seguintes.

In [ ]:
def extrair_features_dataset(df, nome):
    """Extrai LBP, HSV e rótulos na mesma ordem das linhas dos metadados."""
    inicio = perf_counter()
    print(f"{nome}: extraindo características de {len(df)} imagens...")

    X_lbp = np.empty((len(df), LBP_POINTS + 2), dtype=np.float32)
    X_hsv = np.empty((len(df), math.prod(HSV_BINS)), dtype=np.float32)
    y = df["class_id"].to_numpy(dtype=np.int64) - 1

    for i, caminho_relativo in enumerate(df["image_path"]):
        imagem = carregar_imagem(PROCESSED_DIR / caminho_relativo)
        X_lbp[i] = extrair_lbp(imagem)
        X_hsv[i] = extrair_hsv(imagem)

    print(f"{nome}: extração concluída ({perf_counter() - inicio:.2f} s).")
    return X_lbp, X_hsv, y

### Execução da extração

Os descritores foram extraídos utilizando os mesmos parâmetros nos conjuntos de treino, validação e teste. Ao final, foram geradas as matrizes de características que serão reutilizadas nos experimentos com os classificadores tradicionais, garantindo uma comparação consistente entre as diferentes configurações.

In [ ]:
X_train_lbp, X_train_hsv, y_train = extrair_features_dataset(df_train, "train")
X_val_lbp, X_val_hsv, y_val = extrair_features_dataset(df_val, "val")
X_test_lbp, X_test_hsv, y_test = extrair_features_dataset(df_test, "test")

### Combinação dos descritores

Foram preparados três conjuntos de características para os experimentos:

| Características | Atributos | Informação representada |
|---|---:|---|
| LBP | 10 | Textura |
| HSV | 64 | Cor |
| LBP + HSV | 74 | Textura e cor |

A combinação `LBP + HSV` foi obtida pela concatenação dos vetores de características de cada imagem. Dessa forma, o modelo passa a utilizar informações de textura e cor ao mesmo tempo, permitindo avaliar se os dois descritores se complementam na classificação das espécies.

In [ ]:
def combinar_features(X_lbp, X_hsv):
    return {
        "LBP": X_lbp,
        "HSV": X_hsv,
        "LBP + HSV": np.hstack([X_lbp, X_hsv]),
    }


features_train = combinar_features(X_train_lbp, X_train_hsv)
features_val = combinar_features(X_val_lbp, X_val_hsv)
features_test = combinar_features(X_test_lbp, X_test_hsv)

print("Número de atributos:", {nome: X.shape[1] for nome, X in features_train.items()})

### Classificadores e configuração

Foram utilizados dois classificadores tradicionais: **LinearSVC** e **Random Forest**. O LinearSVC foi escolhido como uma abordagem linear de referência e utilizou `StandardScaler`, ajustado somente com os dados de treinamento de cada experimento. Como o número de amostras é maior que o número de características, foi utilizada a configuração `dual=False`.

O **Random Forest** foi utilizado para avaliar uma abordagem capaz de representar relações não lineares entre as características de cor e textura. Foram usadas 100 árvores, profundidade máxima de 20 e pelo menos 2 amostras por folha, buscando limitar a complexidade e o custo de treinamento.

Os parâmetros dos classificadores foram mantidos fixos durante os experimentos para permitir uma comparação mais consistente entre as diferentes combinações de características.

In [ ]:
def criar_modelo(nome):
    if nome == "LinearSVC":
        return make_pipeline(
            StandardScaler(),
            LinearSVC(C=1.0, dual=False, tol=1e-3, max_iter=2000, random_state=SEED),
        )
    if nome == "Random Forest":
        return RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1,
            random_state=SEED,
        )
    raise ValueError(f"Classificador desconhecido: {nome}")


### Métricas de avaliação

O desempenho dos modelos foi avaliado utilizando **acurácia, precisão, recall e F1-score**. Para precisão, recall e F1 foi utilizada a média **macro**, que atribui o mesmo peso a cada uma das 200 espécies.

O **F1 macro no conjunto de validação** foi adotado como principal critério para selecionar a melhor configuração, por representar de forma mais equilibrada o desempenho entre todas as classes.

In [ ]:
def avaliar(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


### Treinamento e validação

Foram realizados quatro experimentos utilizando a mesma divisão entre treino e validação:

| Classificador | Características | Objetivo |
|---|---|---|
| LinearSVC | LBP | Avaliar o uso isolado de textura |
| LinearSVC | HSV | Avaliar o uso isolado de cor |
| LinearSVC | LBP + HSV | Avaliar a combinação de textura e cor |
| Random Forest | LBP + HSV | Comparar dois classificadores utilizando o mesmo conjunto de características |

Os três experimentos com SVM formam o estudo de ablação, permitindo comparar a contribuição de cada descritor individualmente e em conjunto. Todos os modelos foram treinados com o conjunto de treino e avaliados sobre o mesmo conjunto de validação.

In [ ]:
experimentos = [
    ("LinearSVC", "LBP"),
    ("LinearSVC", "HSV"),
    ("LinearSVC", "LBP + HSV"),
    ("Random Forest", "LBP + HSV"),
]

In [ ]:
resultados = []

for nome_classificador, nome_features in experimentos:
    print(f"Treinando {nome_classificador} com {nome_features}...")
    modelo = criar_modelo(nome_classificador)

    inicio = perf_counter()
    modelo.fit(features_train[nome_features], y_train)
    tempo_treino = perf_counter() - inicio

    y_pred_val = modelo.predict(features_val[nome_features])
    metricas = avaliar(y_val, y_pred_val)
    resultados.append(
        {
            "Features": nome_features,
            "Classificador": nome_classificador,
            "tempo_treino_s": tempo_treino,
            **metricas,
        }
    )
    print(f"Concluído em {tempo_treino:.2f} s | F1 validação: {metricas['f1']:.4f}")

### Comparação e seleção do modelo

Os resultados dos experimentos foram comparados utilizando as métricas obtidas no conjunto de validação e o tempo de treinamento de cada configuração. O **F1 macro** foi utilizado como principal critério de seleção, por considerar igualmente o desempenho das 200 classes.

A configuração com maior F1 macro na validação foi escolhida para a etapa seguinte de avaliação no conjunto de teste.


In [ ]:
df_resultados = pd.DataFrame(resultados)
display(df_resultados.sort_values("f1", ascending=False))

In [ ]:
melhor_resultado = df_resultados.loc[df_resultados["f1"].idxmax()]
nome_features = melhor_resultado["Features"]
nome_classificador = melhor_resultado["Classificador"]

print(f"Melhor configuração: {nome_classificador} com {nome_features}")
print(f"F1 macro na validação: {melhor_resultado['f1']:.4f}")

### Treinamento final e avaliação no teste

Após a escolha da melhor configuração, os conjuntos de treino e validação foram unidos para treinar o modelo final. Dessa forma, todas as imagens disponíveis para desenvolvimento foram aproveitadas, enquanto o conjunto de teste permaneceu separado para a avaliação final.

Caso o SVM fosse selecionado, sua pipeline seria ajustada novamente utilizando treino + validação antes da avaliação no teste.

In [ ]:
X_train_final = np.vstack([features_train[nome_features], features_val[nome_features]])
y_train_final = np.concatenate([y_train, y_val])
modelo_final = criar_modelo(nome_classificador)

inicio = perf_counter()
modelo_final.fit(X_train_final, y_train_final)
tempo_treino_final = perf_counter() - inicio

### Resultado final

O modelo selecionado foi avaliado uma única vez no conjunto de teste, utilizando as mesmas métricas adotadas nas etapas anteriores. Essa avaliação permite verificar o desempenho do modelo em dados não utilizados durante a seleção e estimar sua capacidade de generalização.

In [ ]:
y_pred_test = modelo_final.predict(features_test[nome_features])
resultado_teste = avaliar(y_test, y_pred_test)

df_resultado_teste = pd.DataFrame(
    [
        {
            "Features": nome_features,
            "Classificador": nome_classificador,
            "tempo_treino_s": tempo_treino_final,
            **resultado_teste,
        }
    ]
)
print("Resultado final no conjunto de teste:")
display(df_resultado_teste)

### Resultados do Método Tradicional

No estudo de ablação com SVM, o F1 macro foi de **1,01% com LBP**, **1,31% com HSV** e **2,22% com LBP + HSV**. A combinação dos dois descritores apresentou o melhor resultado entre as três configurações.

Com `LBP + HSV`, o **Random Forest** apresentou desempenho superior ao SVM, alcançando **7,01% de acurácia** e **5,83% de F1 macro** na validação. Por isso, essa configuração foi escolhida para a avaliação final.

No conjunto de teste, o Random Forest obteve **6,58% de acurácia**, **5,95% de precisão macro**, **6,66% de recall macro** e **5,79% de F1 macro**. Os resultados ficaram próximos aos obtidos na validação, indicando um comportamento consistente do modelo.

Mesmo assim, o desempenho final foi baixo. O CUB-200-2011 é um problema de classificação **fine-grained**, com 200 espécies visualmente semelhantes e diferenças concentradas em pequenos detalhes, como plumagem, bico e formato das asas. Descritores tradicionais como LBP e HSV não conseguem representar bem toda essa complexidade visual.

Dessa forma, o método tradicional foi utilizado como **linha de base** para comparação com modelos mais avançados. Redes neurais convolucionais e técnicas de **transfer learning** são mais adequadas para esse tipo de tarefa, pois conseguem aprender características visuais mais complexas diretamente das imagens.

# 7. Aumento de dados

O aumento de dados foi realizado de forma **offline** e somente no conjunto de treino. Para cada imagem das variantes `full` e `bbox`, foram geradas **5 novas versões**, sempre a partir da imagem de origem. Essa estratégia facilita a inspeção das imagens geradas, mantém os metadados organizados e permite reproduzir o processo.

O conjunto de treino possui **9.590 imagens de origem**, sendo 4.795 `full` e 4.795 `bbox`. Com 5 aumentos por imagem, foram geradas **47.950 novas imagens**, totalizando **57.540 imagens de treino**.

O aumento foi aplicado somente ao treino para aumentar a variedade dos exemplos apresentados ao modelo durante o aprendizado. Os conjuntos de validação e teste permaneceram sem augmentation, permitindo uma avaliação mais confiável sobre imagens não modificadas.

Como as classes apresentam distribuição relativamente equilibrada, foi utilizada a mesma quantidade de aumentos para todas as imagens, mantendo a proporção entre as espécies.

### Configuração e seleção das origens

Foram selecionadas apenas as imagens de treino nas variantes `full` e `bbox`. O parâmetro `NUM_AUGMENTACOES` define quantas novas versões serão geradas para cada imagem de origem, permitindo controlar diretamente o tamanho final do conjunto aumentado.

In [ ]:
SEED = 42
NUM_AUGMENTACOES = 5
AUG_DIR = PROCESSED_DIR / "train" / "aug"

df_train_aug = df_processado.loc[
    (df_processado["split"] == "Treino")
    & df_processado["variant"].isin(["full", "bbox"])
].copy()

total_aug_esperado = len(df_train_aug) * NUM_AUGMENTACOES

print(f"Origens de treino: {len(df_train_aug):,}")
print(f"Novas imagens por origem: {NUM_AUGMENTACOES}")
print(f"Total esperado de novas imagens: {total_aug_esperado:,}")
display(df_train_aug["variant"].value_counts().rename("Origens"))

### Transformações e justificativa

Foi utilizada a biblioteca **Albumentations** para gerar variações moderadas das imagens, preservando características importantes para distinguir as espécies. Todas as imagens aumentadas foram mantidas em **384 × 384 pixels**.

| Transformação | Aplicação | Justificativa |
|---|---|---|
| Espelhamento horizontal, transformação afim ou perspectiva | Uma transformação geométrica por versão | Introduzir variações de orientação, posição e enquadramento da ave. |
| Rotação de até 10°, escala entre 90% e 110% e deslocamento de até 5% | Quando aplicada a transformação afim | Simular pequenas mudanças de pose e distância sem distorcer excessivamente a imagem. |
| Brilho e contraste de até 10% | Probabilidade de 30% | Representar diferentes condições de iluminação, preservando as cores da plumagem. |
| Desfoque, ruído ou compressão JPEG | Probabilidade de 20% | Tornar o modelo mais robusto a diferenças de foco, ruído e qualidade das imagens. |

As transformações foram mantidas leves porque o **CUB-200-2011 é um problema de classificação fine-grained**, no qual pequenos detalhes de cor, textura e forma são importantes para diferenciar espécies visualmente semelhantes. Por isso, foram evitadas alterações mais agressivas que poderiam modificar essas características.

In [ ]:
def criar_augmentacao():
    return A.Compose(
        [
            # Uma transformação geométrica em cada versão.
            A.OneOf(
                [
                    A.HorizontalFlip(p=1.0),
                    A.Affine(
                        scale=(0.90, 1.10),
                        translate_percent=(-0.05, 0.05),
                        rotate=(-10, 10),
                        border_mode=cv2.BORDER_REFLECT_101,
                        p=1.0,
                    ),
                    A.Perspective(scale=(0.02, 0.05), p=1.0),
                ],
                p=1.0,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.10, contrast_limit=0.10, p=0.3
            ),
            A.OneOf(
                [
                    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                    A.MotionBlur(blur_limit=5, p=1.0),
                    A.GaussNoise(p=1.0),
                    A.ImageCompression(quality_range=(75, 95), p=1.0),
                ],
                p=0.2,
            ),
        ],
        seed=SEED,
    )

### Geração das cinco versões

Para cada imagem de origem foram geradas cinco versões aumentadas utilizando a mesma pipeline de transformações. As novas imagens foram criadas sempre a partir da imagem original processada, evitando o acúmulo de transformações entre versões.
Os arquivos foram nomeados indicando a variante de origem e o número do aumento, como nome_full_aug_01.jpg até nome_full_aug_05.jpg. Dessa forma, as imagens geradas a partir de full e bbox permanecem separadas e fáceis de identificar. Em novas execuções, os mesmos nomes de arquivo são reutilizados, mantendo a organização do conjunto.

In [ ]:
AUG_DIR.mkdir(parents=True, exist_ok=True)
augmentacao = criar_augmentacao()
registros_aug = []

for _, registro in tqdm(
    df_train_aug.iterrows(),
    total=len(df_train_aug),
    desc=f"Gerando {NUM_AUGMENTACOES} versões por origem",
):
    caminho_origem = PROCESSED_DIR / registro["image_path"]

    with Image.open(caminho_origem) as imagem:
        imagem_origem = np.array(imagem.convert("RGB"))

    origem_variant = registro["variant"]
    nome_original = Path(registro["image_path"]).stem
    pasta_classe = AUG_DIR / registro["class_name"]
    pasta_classe.mkdir(parents=True, exist_ok=True)

    for indice_aug in range(1, NUM_AUGMENTACOES + 1):
        # Cada aumento parte da mesma imagem de origem.
        imagem_aug = augmentacao(image=imagem_origem)["image"]
        nome_aug = f"{nome_original}_{origem_variant}_aug_{indice_aug:02d}.jpg"
        caminho_destino = pasta_classe / nome_aug
        Image.fromarray(imagem_aug).save(caminho_destino, quality=95)

        registros_aug.append(
            {
                "image_id": registro["image_id"],
                "class_id": registro["class_id"],
                "class_name": registro["class_name"],
                "split": "Treino",
                "variant": "aug",
                "source_variant": origem_variant,
                "augmentation_index": indice_aug,
                "image_path": str(caminho_destino.relative_to(PROCESSED_DIR)),
                "source_path": registro["image_path"],
            }
        )

df_aug = pd.DataFrame(registros_aug)
print(f"Total de imagens aumentadas: {len(df_aug):,}")

### Verificação visual

In [ ]:
amostras = df_train_aug.sample(n=min(2, len(df_train_aug)), random_state=SEED)

fig, axes = plt.subplots(
    len(amostras),
    NUM_AUGMENTACOES + 1,
    figsize=(3 * (NUM_AUGMENTACOES + 1), 3 * len(amostras)),
    squeeze=False,
)

for linha, (_, origem) in enumerate(amostras.iterrows()):
    with Image.open(PROCESSED_DIR / origem["image_path"]) as imagem:
        axes[linha, 0].imshow(imagem)
    axes[linha, 0].set_title(f"Origem ({origem['variant']})")

    versoes = df_aug.loc[df_aug["source_path"] == origem["image_path"]].sort_values(
        "augmentation_index"
    )

    for coluna, (_, registro) in enumerate(versoes.iterrows(), start=1):
        with Image.open(PROCESSED_DIR / registro["image_path"]) as imagem:
            axes[linha, coluna].imshow(imagem)
        axes[linha, coluna].set_title(f"Aumento {registro['augmentation_index']}")

    for eixo in axes[linha]:
        eixo.axis("off")

plt.tight_layout()
plt.show()

# 8. Abordagem de aprendizado profundo

### Modelos e configuração

Foram utilizadas as arquiteturas MobileNetV2 e EfficientNetV2S, ambas pré-treinadas no ImageNet. A escolha pelo transfer learning permite aproveitar características visuais já aprendidas em um conjunto maior de imagens, o que é adequado ao CUB-200-2011, que possui poucas imagens por espécie. A MobileNetV2 foi escolhida como uma alternativa mais leve, enquanto a EfficientNetV2S permite avaliar uma arquitetura de maior capacidade.
As imagens foram mantidas em 384 × 384 pixels, com batch size igual a 4. O treinamento foi dividido em duas etapas. Primeiro, apenas o classificador final foi treinado por até 8 épocas, utilizando Adam com taxa de aprendizado de 0,001. Depois, foi realizado o fine-tuning das últimas 30 camadas por até 15 épocas, com taxa de aprendizado reduzida para 0,00001. Também foram utilizados ReduceLROnPlateau e EarlyStopping, ambos monitorando a perda de validação.
Cada arquitetura foi treinada em duas condições, totalizando quatro experimentos:
- Sem aumento: apenas as imagens full do conjunto de treino.
- Com aumento: imagens full, recortes bbox e imagens aumentadas offline na Etapa 7.
Os conjuntos de validação e teste utilizaram somente imagens full e permaneceram iguais em todos os experimentos. Dessa forma, foi possível comparar não apenas as arquiteturas, mas também o impacto do uso de recortes e aumento de dados no treinamento.

In [ ]:
IMG_SIZE = (384, 384)
NUM_CLASSES = 200
BATCH_SIZE = 16
SEED = 42
EPOCHS_CABECA = 30
EPOCHS_AJUSTE = 60   
keras.utils.set_random_seed(SEED)

df_treino_original_dl = df_processado[df_processado["split"] == "Treino"]
conjuntos_treino_dl = {
    "Sem aumento": df_treino_original_dl[df_treino_original_dl["variant"] == "full"],
    "Com aumento": pd.concat([df_treino_original_dl, df_aug], ignore_index=True),
}
df_val_dl = df_processado[
    (df_processado["split"] == "Validação") & (df_processado["variant"] == "full")
].reset_index(drop=True)
df_teste_dl = df_processado[
    (df_processado["split"] == "Teste") & (df_processado["variant"] == "full")
].reset_index(drop=True)

for aumento, df in conjuntos_treino_dl.items():
    print(f"Treino {aumento.lower()}: {len(df)} imagens")
print("Validação:", len(df_val_dl), "| Teste:", len(df_teste_dl))

### Leitura das imagens

As imagens foram carregadas em lotes durante o treinamento, evitando manter todo o conjunto na memória ao mesmo tempo. O embaralhamento foi aplicado apenas ao conjunto de treino, enquanto os conjuntos de validação e teste mantiveram sua ordem original.
Os rótulos das 200 espécies também foram convertidos do intervalo 1–200 para 0–199, formato utilizado pelas camadas de saída dos modelos.

In [ ]:
def ler_imagem_dl(caminho, rotulo):
    imagem = tf.io.decode_jpeg(tf.io.read_file(caminho), channels=3)
    return tf.image.resize(imagem, IMG_SIZE), rotulo


def criar_dataset_dl(df, treino=False):
    caminhos = [str(PROCESSED_DIR / p) for p in df["image_path"]]
    rotulos = df["class_id"].to_numpy(dtype=np.int32) - 1
    dataset = tf.data.Dataset.from_tensor_slices((caminhos, rotulos))
    if treino:
        dataset = dataset.shuffle(len(df), seed=SEED)
    return (
        dataset.map(ler_imagem_dl, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


ds_val_dl = criar_dataset_dl(df_val_dl)
ds_teste_dl = criar_dataset_dl(df_teste_dl)
y_val_dl = df_val_dl["class_id"].to_numpy() - 1
y_teste_dl = df_teste_dl["class_id"].to_numpy() - 1

### Construção dos modelos

Nos dois modelos, o classificador original foi substituído por uma camada de Dropout seguida de uma camada softmax com 200 saídas, uma para cada espécie.
A MobileNetV2 utiliza normalização dos pixels para o intervalo [-1, 1], enquanto a EfficientNetV2S realiza o pré-processamento internamente. Durante o fine-tuning, as camadas de BatchNormalization permaneceram congeladas, pois o treinamento foi realizado com lotes pequenos, evitando alterações instáveis nessas estatísticas.

In [ ]:
def criar_modelo_dl(nome, pesos="imagenet"):
    arquiteturas = {
        "MobileNetV2": keras.applications.MobileNetV2,
        "EfficientNetV2S": keras.applications.EfficientNetV2S,
    }
    base = arquiteturas[nome](
        include_top=False, weights=pesos, input_shape=(*IMG_SIZE, 3), pooling="avg"
    )
    base.trainable = False
    entrada = keras.Input(shape=(*IMG_SIZE, 3))
    x = entrada
    if nome == "MobileNetV2":
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x, training=False)
    x = layers.Dropout(0.3)(x)
    saida = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(entrada, saida), base

### Treinamento, seleção e comparação dos aumentos

Os modelos foram treinados individualmente, mantendo em cada fase os pesos com menor perda de validação. Após o fine-tuning, foram calculadas as mesmas métricas utilizadas na Etapa 6.
A melhor configuração foi escolhida com base no F1 macro da validação, considerando tanto a arquitetura quanto o uso ou não de aumento de dados. A tabela de ablação apresenta a diferença de desempenho entre os treinamentos com e sem augmentation, em pontos percentuais, permitindo avaliar o impacto dessa estratégia.

In [ ]:
nomes_modelos_dl = ["MobileNetV2", "EfficientNetV2S"]
experimentos_dl = [
    (nome, aumento) for nome in nomes_modelos_dl for aumento in conjuntos_treino_dl
]
pesos_modelos_dl = {}
historicos_dl = {}
resultados_val_dl = []

for experimento in experimentos_dl:
    nome, aumento = experimento
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    ds_treino_dl = criar_dataset_dl(conjuntos_treino_dl[aumento], treino=True)
    modelo_dl, base_dl = criar_modelo_dl(nome)
    historicos_fases = []

    for fase, epocas, taxa in [
        ("Classificador", EPOCHS_CABECA, 1e-3),
        ("Ajuste fino", EPOCHS_AJUSTE, 1e-5),
    ]:
        print(f"\n{nome} — {aumento} — {fase}")
        if fase == "Ajuste fino":
            base_dl.trainable = True
            for camada in base_dl.layers[:-30]:
                camada.trainable = False
            for camada in base_dl.layers[-30:]:
                camada.trainable = not isinstance(camada, layers.BatchNormalization)

        modelo_dl.compile(
            optimizer=keras.optimizers.Adam(learning_rate=taxa),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        historico = modelo_dl.fit(
            ds_treino_dl,
            validation_data=ds_val_dl,
            epochs=epocas,
            verbose=2,
            shuffle=False,
            callbacks=[
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7
                ),
                keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=4, restore_best_weights=True
                ),
            ],
        )
        historicos_fases.append(pd.DataFrame(historico.history))

    predicoes_val = modelo_dl.predict(ds_val_dl, verbose=0).argmax(axis=1)
    resultados_val_dl.append(
        {"Modelo": nome, "Aumento": aumento, **avaliar(y_val_dl, predicoes_val)}
    )
    pesos_modelos_dl[experimento] = modelo_dl.get_weights()
    historicos_dl[experimento] = pd.concat(historicos_fases, ignore_index=True)
    del modelo_dl, base_dl, historico

keras.backend.clear_session()
df_validacao_dl = pd.DataFrame(resultados_val_dl)
display(df_validacao_dl)
df_ablacao_dl = df_validacao_dl.pivot(index="Modelo", columns="Aumento", values="f1")
df_ablacao_dl["Ganho F1 (p.p.)"] = 100 * (
    df_ablacao_dl["Com aumento"] - df_ablacao_dl["Sem aumento"]
)
display(df_ablacao_dl)

melhor_resultado_dl = df_validacao_dl.loc[df_validacao_dl["f1"].idxmax()]
melhor_nome_dl = melhor_resultado_dl["Modelo"]
melhor_aumento_dl = melhor_resultado_dl["Aumento"]
melhor_experimento_dl = (melhor_nome_dl, melhor_aumento_dl)
print("Melhor modelo na validação:", melhor_nome_dl, "—", melhor_aumento_dl)

### Curvas de aprendizado

As curvas de perda foram utilizadas para acompanhar a evolução do treinamento e da validação, além de identificar possíveis sinais de overfitting. Para cada modelo, as épocas de fine-tuning foram apresentadas em sequência às épocas iniciais de treinamento, facilitando a visualização do comportamento do modelo ao longo das duas fases.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, experimento in zip(axes.flat, experimentos_dl):
    nome, aumento = experimento
    historico = historicos_dl[experimento]
    epocas = np.arange(1, len(historico) + 1)
    ax.plot(epocas, historico["loss"], label="Treino")
    ax.plot(epocas, historico["val_loss"], label="Validação")
    ax.set(title=f"{nome} — {aumento}", xlabel="Época", ylabel="Perda")
    ax.legend()
plt.tight_layout()
plt.show()

### Avaliação no teste e comparação

Após selecionar as melhores configurações com base na validação, os quatro experimentos foram avaliados no conjunto de teste. Foram comparadas as métricas de acurácia, precisão, recall e F1 macro, incluindo também o melhor resultado da abordagem tradicional da Etapa 6.
Todos os modelos foram avaliados utilizando as mesmas imagens de teste. O método tradicional foi retreinado com treino + validação, enquanto as redes neurais utilizaram apenas o conjunto de treino. Os resultados do teste foram usados somente para a avaliação final e não influenciaram a escolha do modelo salvo.

In [ ]:
resultados_teste_dl = []
predicoes_dl = {}

for experimento in experimentos_dl:
    nome, aumento = experimento
    keras.backend.clear_session()
    modelo_dl, base_dl = criar_modelo_dl(nome, pesos=None)
    modelo_dl.set_weights(pesos_modelos_dl[experimento])
    predicoes_dl[experimento] = modelo_dl.predict(ds_teste_dl, verbose=1).argmax(axis=1)
    resultados_teste_dl.append(
        {
            "Modelo": nome,
            "Aumento": aumento,
            **avaliar(y_teste_dl, predicoes_dl[experimento]),
        }
    )
    del modelo_dl, base_dl

keras.backend.clear_session()
df_teste_resultados_dl = pd.DataFrame(resultados_teste_dl)
tradicional_dl = df_resultado_teste[
    ["Classificador", "accuracy", "precision", "recall", "f1"]
].rename(columns={"Classificador": "Modelo"})
tradicional_dl.insert(1, "Aumento", "Sem aumento")
df_comparacao_dl = pd.concat(
    [tradicional_dl, df_teste_resultados_dl], ignore_index=True
)
display(df_comparacao_dl)

### Análise qualitativa

Foram apresentados quatro exemplos de classificações corretas do modelo. Para a análise dos erros, foram selecionadas duas predições incorretas e comparadas com imagens de referência das respectivas classes corretas. Essa comparação permite observar visualmente características que podem ter contribuído para a confusão do modelo, como semelhança entre espécies, pose, plumagem e influência do fundo.

In [ ]:
predicoes = predicoes_dl[melhor_experimento_dl]

acertos = np.flatnonzero(predicoes == y_teste_dl)

rng = np.random.default_rng(SEED)

indices_acertos = rng.choice(acertos, size=min(4, len(acertos)), replace=False)

nomes_classes_dl = df_processado.set_index("class_id")["class_name"].to_dict()

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, indice in zip(axes.flat, indices_acertos):
    registro = df_teste_dl.iloc[indice]

    with Image.open(PROCESSED_DIR / registro["image_path"]) as imagem:
        ax.imshow(imagem)

    classe_real = registro["class_name"]

    ax.set_title(f"Real: {classe_real}\nPrevista: {classe_real}", fontsize=9)

    ax.axis("off")

for ax in axes.flat[len(indices_acertos) :]:
    ax.axis("off")

fig.suptitle(f"Exemplos de acertos — {melhor_nome_dl}", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
erros = np.flatnonzero(predicoes != y_teste_dl)

indices_erros = rng.choice(erros, size=min(2, len(erros)), replace=False)

df_referencia = df_processado[
    (df_processado["split"] == "Treino") & (df_processado["variant"] == "full")
].copy()

fig, axes = plt.subplots(2, 2, figsize=(9, 9))

for linha, indice in enumerate(indices_erros):
    registro = df_teste_dl.iloc[indice]

    classe_real_id = y_teste_dl[indice] + 1

    classe_predita_id = predicoes[indice] + 1

    classe_real = nomes_classes_dl[classe_real_id]

    classe_predita = nomes_classes_dl[classe_predita_id]

    # Imagem classificada incorretamente
    with Image.open(PROCESSED_DIR / registro["image_path"]) as imagem:
        axes[linha, 0].imshow(imagem)

    axes[linha, 0].set_title(
        f"Erro do modelo\nReal: {classe_real}\nPrevista: {classe_predita}", fontsize=9
    )

    axes[linha, 0].axis("off")

    # Exemplo da classe correta
    candidatos = df_referencia[df_referencia["class_id"] == classe_real_id]

    referencia = candidatos.sample(1, random_state=SEED + linha).iloc[0]

    with Image.open(PROCESSED_DIR / referencia["image_path"]) as imagem:
        axes[linha, 1].imshow(imagem)

    axes[linha, 1].set_title(f"Exemplo da classe correta\n{classe_real}", fontsize=9)

    axes[linha, 1].axis("off")


fig.suptitle(f"Análise de erros — {melhor_nome_dl}", fontsize=14)

plt.tight_layout()
plt.show()

### Salvamento do melhor modelo

O modelo com melhor F1 macro na validação foi salvo em data/models/melhor_modelo.keras. O arquivo armazena a arquitetura e os pesos treinados, permitindo reutilizar o modelo posteriormente com keras.models.load_model.
O modelo recebe como entrada imagens RGB de 384 × 384 pixels, mantendo os valores dos pixels no intervalo [0, 255], conforme o pré-processamento utilizado durante o treinamento.

In [ ]:
keras.backend.clear_session()
modelo_final_dl, base_final_dl = criar_modelo_dl(melhor_nome_dl, pesos=None)
modelo_final_dl.set_weights(pesos_modelos_dl[melhor_experimento_dl])
caminho_modelo_dl = Path("./data/models/melhor_modelo.keras")
caminho_modelo_dl.parent.mkdir(parents=True, exist_ok=True)
modelo_final_dl.save(caminho_modelo_dl)
print("Modelo salvo:", melhor_nome_dl, "—", melhor_aumento_dl)
print("Arquivo:", caminho_modelo_dl)

# Resumo

O projeto permitiu comparar uma abordagem tradicional de classificação de imagens com modelos de deep learning aplicados ao CUB-200-2011. A linha de base com LBP, HSV, SVM e Random Forest apresentou desempenho limitado, enquanto os modelos de deep learning conseguiram representar melhor os detalhes visuais das aves e alcançaram resultados superiores.

Entre os principais pontos positivos do trabalho estão a organização dos dados, a separação entre treino, validação e teste, o uso das bounding boxes, o aumento de dados offline e a comparação entre diferentes estratégias de classificação. O projeto também ajudou a perceber, na prática, como escolhas de pré-processamento, representação das imagens e arquitetura do modelo influenciam diretamente os resultados.

Se o trabalho fosse realizado novamente, eu dedicaria menos tempo às abordagens tradicionais e ampliaria os experimentos com deep learning, principalmente em diferentes estratégias de fine-tuning, aumento de dados e comparação entre arquiteturas.
De forma geral, o projeto contribuiu para consolidar conhecimentos de visão computacional e aprendizado de máquina, principalmente em preparação de dados, avaliação de modelos e transfer learning. Além disso, o desenvolvimento completo do pipeline aproximou o conteúdo da disciplina de situações reais, em que é necessário testar alternativas, lidar com limitações computacionais e justificar as decisões tomadas.